# DeBERTa-NLI + ModernBERT-large Fine-tune Combo

Notebook combo đúng 2 stage:

1. **DeBERTa-v3-large-MNLI**: chấm direction `A -> B` vs `B -> A`.
2. **ModernBERT-large fine-tune**: fine-tune binary prerequisite classifier rồi chấm strength.

Nếu có dataset thật, upload `prereq_train_pairs.jsonl` và optional `prereq_val_pairs.jsonl`. Nếu không có, notebook bootstrap bằng P5 labels: `clean_candidate_edges` = positive, `pruned_edges` = negative.

In [ ]:
# Input
DRIVE_URL = ""  # Optional Google Drive link to edge_scoring_input_bundle_for_combo.json or zip containing it.
INPUT_PATH = "/content/edge_scoring_input_bundle_for_combo.json"

# Models
DEBERTA_MODEL_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
MODERNBERT_MODEL_ID = "answerdotai/ModernBERT-large"

# Fine-tune data. If TRAIN_JSONL exists, notebook uses it. Otherwise bootstrap from P5 labels.
TRAIN_JSONL = "/content/prereq_train_pairs.jsonl"
VAL_JSONL = "/content/prereq_val_pairs.jsonl"
BOOTSTRAP_FROM_P5_IF_NO_TRAIN_FILE = True

# Runtime knobs
MAX_LENGTH_NLI = 512
MAX_LENGTH_FT = 512
NLI_BATCH_SIZE = 8
FT_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
EPOCHS = 4
LR = 2e-4
USE_LORA = True
OUTPUT_PATH = "/content/deberta_modernbert_finetune_combo_scores.json"

In [ ]:
!pip -q install -U "transformers>=4.57.0" accelerate gdown safetensors sentencepiece protobuf peft scikit-learn

In [ ]:
import gc
import json
import math
import random
import re
import shutil
import zipfile
from pathlib import Path
from typing import Any

import requests
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def extract_drive_file_id(url: str) -> str:
    patterns = [r"/file/d/([A-Za-z0-9_-]+)", r"[?&]id=([A-Za-z0-9_-]+)"]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Cannot extract Google Drive file id from: {url}")

def download_from_drive(file_id: str, output_path: Path) -> None:
    import gdown
    result = gdown.download(id=file_id, output=str(output_path), quiet=False)
    if result and output_path.exists() and output_path.stat().st_size > 0:
        return
    session = requests.Session()
    url = "https://drive.google.com/uc?export=download"
    response = session.get(url, params={"id": file_id}, stream=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break
    if token:
        response = session.get(url, params={"id": file_id, "confirm": token}, stream=True)
    response.raise_for_status()
    with output_path.open("wb") as handle:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                handle.write(chunk)

def load_payload() -> dict[str, Any]:
    path = Path(INPUT_PATH)
    if path.exists() and path.stat().st_size > 0:
        print("Loading input:", path)
        return json.loads(path.read_text(encoding="utf-8"))
    if not DRIVE_URL.strip():
        raise FileNotFoundError(f"Upload input bundle to {INPUT_PATH} or set DRIVE_URL")
    download_path = Path("/content/edge_scoring_drive_input")
    extract_dir = Path("/content/edge_scoring_drive_extract")
    if download_path.exists():
        download_path.unlink()
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    file_id = extract_drive_file_id(DRIVE_URL)
    print("Drive file id:", file_id)
    download_from_drive(file_id, download_path)
    if zipfile.is_zipfile(download_path):
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(download_path) as zf:
            zf.extractall(extract_dir)
        json_files = sorted(extract_dir.rglob("*.json"))
        if not json_files:
            raise FileNotFoundError("Zip has no JSON files")
        input_path = json_files[0]
    else:
        input_path = download_path
    print("Loading downloaded input:", input_path)
    return json.loads(input_path.read_text(encoding="utf-8"))

payload = load_payload()
print("Top-level keys:", list(payload.keys()))
print("Input stats:", payload.get("input_stats"))

In [ ]:
clean_edges = payload.get("clean_candidate_edges")
pruned_edges = payload.get("pruned_edges")
global_kps = payload.get("global_kps") or payload.get("concepts_kp_global") or []
if not isinstance(clean_edges, list) or not isinstance(pruned_edges, list):
    raise ValueError("Expected clean_candidate_edges[] and pruned_edges[]")

kp_index = {
    kp.get("global_kp_id") or kp.get("kp_id"): kp
    for kp in global_kps
    if isinstance(kp, dict) and (kp.get("global_kp_id") or kp.get("kp_id"))
}

def readable_kp_id(kp_id: str) -> str:
    return re.sub(r"[_-]+", " ", kp_id.removeprefix("kp_")).strip()

def kp_text(kp_id: str) -> str:
    kp = kp_index.get(kp_id, {})
    name = kp.get("name") or kp.get("canonical_name") or readable_kp_id(kp_id)
    desc = kp.get("description") or kp.get("global_description") or ""
    return f"{name}. {desc}".strip()

def pair_text(edge: dict[str, Any], *, reverse: bool = False) -> tuple[str, str]:
    source = edge["target_kp_id"] if reverse else edge["source_kp_id"]
    target = edge["source_kp_id"] if reverse else edge["target_kp_id"]
    return kp_text(source), kp_text(target)

def nli_pair(edge: dict[str, Any], *, reverse: bool = False) -> tuple[str, str]:
    source_text, target_text = pair_text(edge, reverse=reverse)
    premise = f"Source concept: {source_text}
Target concept: {target_text}"
    hypothesis = f"Understanding the source concept is required before understanding the target concept."
    return premise, hypothesis

print("Clean edges:", len(clean_edges))
print("Pruned edges:", len(pruned_edges))
print("KP catalog entries:", len(kp_index))
print("Sample:", nli_pair(clean_edges[0])[0][:500], "
H:", nli_pair(clean_edges[0])[1])

## Stage 1: DeBERTa-NLI Direction Scoring

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device=", device, "dtype=", dtype)
if device == "cuda":
    print("gpu=", torch.cuda.get_device_name(0))

deb_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_ID)
deb_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_MODEL_ID, torch_dtype=dtype).to(device)
deb_model.eval()
print("id2label:", deb_model.config.id2label)

label_to_id = {str(v).lower(): int(k) for k, v in deb_model.config.id2label.items()}
entailment_id = next((i for lab, i in label_to_id.items() if "entail" in lab), None)
contradiction_id = next((i for lab, i in label_to_id.items() if "contrad" in lab), None)
neutral_id = next((i for lab, i in label_to_id.items() if "neutral" in lab), None)
if entailment_id is None or contradiction_id is None:
    raise RuntimeError(f"Cannot find entailment/contradiction labels in {deb_model.config.id2label}")
print("entailment_id", entailment_id, "contradiction_id", contradiction_id, "neutral_id", neutral_id)

def score_nli_pairs(pairs, batch_size=NLI_BATCH_SIZE):
    out = []
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        premises = [p for p, h in batch]
        hypotheses = [h for p, h in batch]
        encoded = deb_tokenizer(premises, hypotheses, padding=True, truncation=True, max_length=MAX_LENGTH_NLI, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = deb_model(**encoded).logits.float()
        probs = F.softmax(logits, dim=-1)
        for row in range(probs.shape[0]):
            entail = probs[row, entailment_id].item()
            contra = probs[row, contradiction_id].item()
            neutral = probs[row, neutral_id].item() if neutral_id is not None else 0.0
            out.append({
                "entailment": entail,
                "contradiction": contra,
                "neutral": neutral,
                "directional_score": entail - contra,
                "strength_score": entail / max(1e-9, entail + contra),
            })
    return out

forward_nli = score_nli_pairs([nli_pair(edge, reverse=False) for edge in clean_edges])
reverse_nli = score_nli_pairs([nli_pair(edge, reverse=True) for edge in clean_edges])
print("NLI scored:", len(forward_nli), len(reverse_nli))

# Free VRAM before ModernBERT fine-tune.
del deb_model
del deb_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Stage 2: Build Fine-tune Dataset

Preferred external train format: JSONL rows with either:

```json
{"source_text": "...", "target_text": "...", "label": 1}
```

or:

```json
{"source_kp_id": "...", "target_kp_id": "...", "label": 0}
```

If no train file exists, notebook bootstraps from P5 labels: clean edges positive, pruned edges negative. This is useful for smoke-test, not final research-grade training.

In [ ]:
def normalize_train_row(row: dict[str, Any]) -> dict[str, Any]:
    label = row.get("label", row.get("is_prereq"))
    if isinstance(label, bool):
        label = int(label)
    if label not in {0, 1}:
        raise ValueError(f"Bad label: {label} row={row}")
    if row.get("source_text") and row.get("target_text"):
        return {"source_text": str(row["source_text"]), "target_text": str(row["target_text"]), "label": int(label)}
    if row.get("source_kp_id") and row.get("target_kp_id"):
        return {"source_text": kp_text(row["source_kp_id"]), "target_text": kp_text(row["target_kp_id"]), "label": int(label)}
    raise ValueError(f"Row needs source_text/target_text or source_kp_id/target_kp_id: {row}")

def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(normalize_train_row(json.loads(line)))
    return rows

def rows_from_p5_bootstrap() -> list[dict[str, Any]]:
    rows = []
    for edge in clean_edges:
        source_text, target_text = pair_text(edge)
        rows.append({"source_text": source_text, "target_text": target_text, "label": 1})
    for edge in pruned_edges:
        source_text, target_text = pair_text(edge)
        rows.append({"source_text": source_text, "target_text": target_text, "label": 0})
    return rows

train_path = Path(TRAIN_JSONL)
val_path = Path(VAL_JSONL)
if train_path.exists():
    train_rows = load_jsonl(train_path)
    val_rows = load_jsonl(val_path) if val_path.exists() else []
    training_source = "external_jsonl"
else:
    if not BOOTSTRAP_FROM_P5_IF_NO_TRAIN_FILE:
        raise FileNotFoundError(f"Missing {TRAIN_JSONL}")
    all_rows = rows_from_p5_bootstrap()
    random.shuffle(all_rows)
    split = max(1, int(0.8 * len(all_rows)))
    train_rows = all_rows[:split]
    val_rows = all_rows[split:]
    training_source = "p5_bootstrap_clean_vs_pruned"

print("training_source", training_source)
print("train", len(train_rows), "pos", sum(r['label'] for r in train_rows), "neg", len(train_rows)-sum(r['label'] for r in train_rows))
print("val", len(val_rows), "pos", sum(r['label'] for r in val_rows), "neg", len(val_rows)-sum(r['label'] for r in val_rows))

## Stage 3: ModernBERT-large Fine-tune

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer

class PairDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        encoded = self.tokenizer(
            row["source_text"],
            row["target_text"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item["labels"] = torch.tensor(row["label"], dtype=torch.long)
        return item

mb_tokenizer = AutoTokenizer.from_pretrained(MODERNBERT_MODEL_ID)
mb_model = AutoModelForSequenceClassification.from_pretrained(
    MODERNBERT_MODEL_ID,
    num_labels=2,
    torch_dtype=dtype,
    attn_implementation="sdpa",
).to(device)

peft_enabled = False
if USE_LORA:
    try:
        from peft import LoraConfig, TaskType, get_peft_model
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules="all-linear",
        )
        mb_model = get_peft_model(mb_model, lora_config)
        peft_enabled = True
        mb_model.print_trainable_parameters()
    except Exception as exc:
        print("LoRA attach failed; fallback to classifier-head tuning:", repr(exc))
        for name, param in mb_model.named_parameters():
            param.requires_grad = name.startswith("classifier") or "score" in name

train_ds = PairDataset(train_rows, mb_tokenizer, MAX_LENGTH_FT)
val_ds = PairDataset(val_rows, mb_tokenizer, MAX_LENGTH_FT) if val_rows else None
train_loader = DataLoader(train_ds, batch_size=FT_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False) if val_ds else None

optimizer = torch.optim.AdamW([p for p in mb_model.parameters() if p.requires_grad], lr=LR)
mb_model.train()

history = []
for epoch in range(1, EPOCHS + 1):
    losses = []
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        out = mb_model(**batch)
        loss = out.loss
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    epoch_row = {"epoch": epoch, "train_loss": sum(losses) / max(1, len(losses))}
    if val_loader:
        mb_model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                labels.extend(batch["labels"].tolist())
                batch = {k: v.to(device) for k, v in batch.items()}
                logits = mb_model(**batch).logits.float()
                preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
        epoch_row.update({
            "val_accuracy": accuracy_score(labels, preds),
            "val_f1": f1_score(labels, preds, zero_division=0),
        })
        mb_model.train()
    history.append(epoch_row)
    print(epoch_row)

print("Fine-tune done. peft_enabled=", peft_enabled)

## Stage 4: Score P5 Clean Edges With Fine-tuned ModernBERT

In [ ]:
def score_modernbert_pairs(edges, *, reverse=False, batch_size=EVAL_BATCH_SIZE):
    mb_model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(edges), batch_size):
            batch_edges = edges[start:start + batch_size]
            src_texts, tgt_texts = [], []
            for edge in batch_edges:
                source_text, target_text = pair_text(edge, reverse=reverse)
                src_texts.append(source_text)
                tgt_texts.append(target_text)
            encoded = mb_tokenizer(
                src_texts,
                tgt_texts,
                truncation=True,
                max_length=MAX_LENGTH_FT,
                padding=True,
                return_tensors="pt",
            ).to(device)
            logits = mb_model(**encoded).logits.float()
            probs = F.softmax(logits, dim=-1)
            for i in range(probs.shape[0]):
                scores.append({
                    "prob_not_prereq": probs[i, 0].item(),
                    "prob_prereq": probs[i, 1].item(),
                    "logit_not_prereq": logits[i, 0].item(),
                    "logit_prereq": logits[i, 1].item(),
                })
    return scores

mb_forward = score_modernbert_pairs(clean_edges, reverse=False)
mb_reverse = score_modernbert_pairs(clean_edges, reverse=True)
print("ModernBERT fine-tuned scored:", len(mb_forward), len(mb_reverse))

## Stage 5: Combine Direction + Strength

In [ ]:
def direction_verdict(margin: float) -> str:
    if margin >= 0.15:
        return "forward"
    if margin <= -0.15:
        return "reverse"
    return "ambiguous"

def combo_verdict(nli_margin: float, strength: float, reverse_strength: float) -> str:
    dv = direction_verdict(nli_margin)
    if dv == "reverse" and reverse_strength >= 0.5:
        return "flip"
    if strength < 0.35 and reverse_strength < 0.35:
        return "prune"
    if dv == "forward" and strength >= 0.5:
        return "keep"
    if strength >= 0.65:
        return "keep_optional"
    return "ambiguous"

scored_edges = []
for edge, fn, rn, fm, rm in zip(clean_edges, forward_nli, reverse_nli, mb_forward, mb_reverse):
    nli_margin = fn["directional_score"] - rn["directional_score"]
    ft_strength = fm["prob_prereq"]
    ft_reverse = rm["prob_prereq"]
    row = dict(edge)
    row.update({
        "deberta_model": DEBERTA_MODEL_ID,
        "deberta_forward_entailment": round(fn["entailment"], 4),
        "deberta_forward_contradiction": round(fn["contradiction"], 4),
        "deberta_reverse_entailment": round(rn["entailment"], 4),
        "deberta_reverse_contradiction": round(rn["contradiction"], 4),
        "deberta_direction_margin": round(nli_margin, 4),
        "deberta_direction_verdict": direction_verdict(nli_margin),
        "modernbert_model": MODERNBERT_MODEL_ID,
        "modernbert_training_source": training_source,
        "modernbert_peft_lora_enabled": peft_enabled,
        "modernbert_ft_edge_strength": round(ft_strength, 4),
        "modernbert_ft_reverse_strength": round(ft_reverse, 4),
        "modernbert_ft_strength_margin": round(ft_strength - ft_reverse, 4),
        "combo_verdict": combo_verdict(nli_margin, ft_strength, ft_reverse),
    })
    scored_edges.append(row)

from collections import Counter
margins = [e["deberta_direction_margin"] for e in scored_edges]
strengths = [e["modernbert_ft_edge_strength"] for e in scored_edges]
output = {
    "run_id": payload.get("run_id", "p5_cs224n_cs231n"),
    "stage_id": "deberta_modernbert_finetune_combo",
    "input_source_has_external_audit_labels": False,
    "deberta_model": DEBERTA_MODEL_ID,
    "modernbert_model": MODERNBERT_MODEL_ID,
    "modernbert_training_source": training_source,
    "modernbert_peft_lora_enabled": peft_enabled,
    "fine_tune_history": history,
    "scored_edges": scored_edges,
    "score_summary": {
        "edge_count": len(scored_edges),
        "combo_verdict_distribution": dict(Counter(e["combo_verdict"] for e in scored_edges)),
        "deberta_direction_verdict_distribution": dict(Counter(e["deberta_direction_verdict"] for e in scored_edges)),
        "deberta_direction_margin_min": round(min(margins), 4),
        "deberta_direction_margin_max": round(max(margins), 4),
        "deberta_direction_margin_mean": round(sum(margins) / len(margins), 4),
        "modernbert_ft_strength_min": round(min(strengths), 4),
        "modernbert_ft_strength_max": round(max(strengths), 4),
        "modernbert_ft_strength_mean": round(sum(strengths) / len(strengths), 4),
    },
}
Path(OUTPUT_PATH).write_text(json.dumps(output, ensure_ascii=False, indent=2) + "
", encoding="utf-8")
print(json.dumps(output["score_summary"], indent=2))
print("Wrote", OUTPUT_PATH)

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except Exception as exc:
    print("files.download unavailable:", exc)
    print("Output path:", OUTPUT_PATH)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_output = Path("/content/drive/MyDrive/deberta_modernbert_finetune_combo_scores.json")
    shutil.copyfile(OUTPUT_PATH, drive_output)
    print("Saved to Google Drive:", drive_output)
except Exception as exc:
    print("Drive save skipped:", exc)